# Notebook 01: Prompt Structure — The PCFT Framework

This notebook teaches you to construct effective prompts using the **Persona-Context-Task-Format** framework. You will compare simple vs. structured prompts and measure output quality across variations.

## Learning Objectives
- Identify the four components of a well-structured prompt
- Compare outputs from simple vs. structured prompts
- Use the Gemini API to test prompt variations
- Evaluate output quality systematically

In [ ]:
import os
import json
import openai


# Initialize Gemini client
# Set your API key: export OPENAI_API_KEY="your-key"
client = openai.OpenAI()

def generate(prompt, system_prompt=None, temperature=0.7, max_tokens=1024):
    """Generate response from Gemini."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=max_tokens
    )
    if system_prompt:
        config.system_instruction = system_prompt
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}],
        config=config
    )
    return response.choices[0].message.content

print("Gemini client initialized.")

## 1. The PCFT Framework

Every effective prompt contains four components:

| Component | Purpose | Example |
|-----------|---------|---------|
| **Persona** | Who the model should act as | "You are a senior data engineer" |
| **Context** | Background information | "The data pipeline processes 10M events/day" |
| **Task** | Specific action to perform | "Identify bottlenecks in the pipeline" |
| **Format** | Output structure | "Respond in JSON with fields: ..." |

## 2. Simple vs. Structured Prompts

Let's compare a naive prompt against a well-structured one.

In [ ]:
# Simple prompt (bad)
simple_prompt = """
Tell me about this code and if there are any problems.

def process_data(data):
    result = []
    for item in data:
        if item['type'] == 'user':
            result.append(item)
    return result
"""

print("=== SIMPLE PROMPT ===")
print(simple_prompt)
print("\n--- Response ---")
response_simple = generate(simple_prompt, temperature=0.7)
print(response_simple)

In [ ]:
# Structured prompt using PCFT (good)
persona = "You are a senior Python developer conducting a code review."

context = """The following function is part of a data processing pipeline
that handles 10M events per day. It filters incoming data to extract
only user-type records."""

task = """Review the code for:
1. Correctness — does it do what it claims?
2. Performance — any efficiency concerns at scale?
3. Edge cases — what happens with empty input, None values, or missing keys?
4. Best practices — type hints, naming, error handling"""

format_spec = """Respond in this exact JSON format:
{
  "correctness": "assessment",
  "performance": "assessment",
  "edge_cases": ["case1", "case2"],
  "best_practices": ["issue1", "issue2"],
  "severity": "low|medium|high",
  "rewritten_code": "improved version"
}"""

code_snippet = """
def process_data(data):
    result = []
    for item in data:
        if item['type'] == 'user':
            result.append(item)
    return result
"""

structured_prompt = f"""
{persona}

{context}

{task}

{format_spec}

--- CODE ---
{code_snippet}
--- END CODE ---
"""

print("=== STRUCTURED PROMPT ===")
print(structured_prompt)
print("\n--- Response ---")
response_structured = generate(structured_prompt, temperature=0.3)
print(response_structured)

### Discussion

Compare the two responses:
- Which response is more actionable?
- Which response has a consistent format?
- Which response could you parse programmatically?

## 3. Impact of Each Component

Let's isolate the effect of each PCFT component by adding them one at a time.

In [ ]:
review_text = """The new dashboard is incredibly slow. It takes 30 seconds to load
and the charts keep freezing. I've reported this twice but nothing changed.
Seriously considering switching to a competitor."""

# Level 1: Task only
prompt_level1 = f"Analyze this feedback: {review_text}"

# Level 2: Task + Format
prompt_level2 = f"""Analyze this customer feedback.
Return: sentiment, category, urgency (low/medium/high)."""

# Level 3: Persona + Task + Format
prompt_level3 = f"""You are a customer success manager.
Analyze this customer feedback.
Return: sentiment, category, urgency (low/medium/high)."""

# Level 4: Persona + Context + Task + Format
prompt_level4 = f"""You are a customer success manager at a SaaS company.
This feedback was received through the support portal.
The customer is on the Enterprise plan ($50K ARR).

Analyze this feedback and recommend an action.

Return JSON:
{\"sentiment\": \"positive|negative|neutral\", \"category\": \"performance|support|product\", \"urgency\": \"low|medium|high\", \"action\": \"specific next step\"}"""

for level, prompt in [("Level 1 (Task only)", prompt_level1),
                       ("Level 2 (Task + Format)", prompt_level2),
                       ("Level 3 (Persona + Task + Format)", prompt_level3),
                       ("Level 4 (Full PCFT)", prompt_level4)]:
    print(f"\n{'='*50}")
    print(f"{level}")
    print(f"{'='*50}")
    result = generate(prompt, temperature=0.3)
    print(result[:300])

## 4. Measuring Output Quality

Let's define quality metrics and compare prompt variations systematically.

In [ ]:
def evaluate_prompt(prompt, label, num_runs=3):
    """Run a prompt multiple times and measure consistency."""
    results = []
    for _ in range(num_runs):
        try:
            response = generate(prompt, temperature=0.3)
            results.append(response)
        except Exception as e:
            results.append(f"ERROR: {e}")

    print(f"\n=== {label} ===")
    print(f"Runs completed: {len([r for r in results if not r.startswith('ERROR')])}/{num_runs}")

    # Check if responses contain JSON
    json_count = sum(1 for r in results if '{' in r and '}' in r)
    print(f"JSON-like responses: {json_count}/{num_runs}")

    # Show first result
    print(f"\nSample output:\n{results[0][:200]}")

    return results

# Test with a classification task
test_input = "This API endpoint returns 500 errors intermittently under high load."

# Vague prompt
vague = f"What is this about? {test_input}"

# Specific prompt
specific = f"""Classify this issue report.
Return JSON: {{\"type\": \"bug|feature|question\", \"severity\": \"low|medium|high\", \"component\": \"api|ui|data\"}}

Issue: {test_input}"""

results_vague = evaluate_prompt(vague, "Vague Prompt")
results_specific = evaluate_prompt(specific, "Specific Prompt")

## 5. Exercises

### Exercise 1: Build a PCFT Prompt

Create a prompt to analyze a git commit message and determine:
- Is this a feature, bugfix, refactor, or docs change?
- What is the risk level (low/medium/high)?
- What testing is needed?

Requirements:
- Include all four PCFT components
- Output must be valid JSON
- Test it with at least 3 different commit messages

In [ ]:
# YOUR CODE HERE
# Step 1: Define persona, context, task, format
# Step 2: Combine into a prompt
# Step 3: Test with different commit messages

commit_messages = [
    "feat(auth): add OAuth2 support for Google login",
    "fix: resolve race condition in connection pool",
    "refactor: extract validation logic into shared utils",
]

# Build your prompt template here
# prompt_template = ...

### Exercise 2: Improve a Bad Prompt

The following prompt produces inconsistent results. Rewrite it using PCFT and explain what you changed.

```text
Summarize this.
The quarterly report shows revenue increased 15% YoY to $4.2M. Operating expenses
decreased 8% due to automation initiatives. Customer acquisition cost dropped from
$120 to $85. Net retention rate improved to 118%. However, churn in the SMB
segment increased to 5.2% from 3.8%. The board approved a $2M Series B extension
for international expansion into APAC markets.
```

In [ ]:
# YOUR CODE HERE
# Rewrite the prompt using PCFT framework
# Test it and compare the output quality

report = """The quarterly report shows revenue increased 15% YoY to $4.2M. Operating expenses
decreased 8% due to automation initiatives. Customer acquisition cost dropped from
$120 to $85. Net retention rate improved to 118%. However, churn in the SMB
segment increased to 5.2% from 3.8%. The board approved a $2M Series B extension
for international expansion into APAC markets."""

# improved_prompt = ...

### Exercise 3: Format Control

Write prompts that force the model to return output in these exact formats:

1. A Python dictionary with specific keys
2. A markdown table
3. A CSV string

Test each one and verify the output is parseable.

In [ ]:
# YOUR CODE HERE
# Write three prompts that enforce specific output formats

sample_data = "Amazon AWS, Microsoft Azure, Google Cloud, Oracle Cloud, IBM Cloud"

# Prompt 1: Python dictionary
# Prompt 2: Markdown table
# Prompt 3: CSV string

## Key Takeaways

1. **Always use all four PCFT components** — Persona, Context, Task, Format
2. **Be explicit about output format** — don't leave it to chance
3. **More structure = more consistency** — structured prompts produce parseable output
4. **Temperature affects quality** — lower temperature for extraction, higher for creative tasks
5. **Test systematically** — run prompts multiple times and measure consistency